In [3]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [6]:
spark=SparkSession.builder \
.appName("Spark Assignment") \
.getOrCreate()

spark

In [7]:
%%writefile dataset.csv
user_id,name,age,category,region,sales,status
1,Amit,25,Electronics,West,500,Active
2,Riya,30,Furniture,East,700,
3,Rohan,17,Electronics,West,300,Active
4,Neha,22,Clothing,North,200,Inactive
5,Amit,25,Electronics,West,500,Active
6,Karan,40,Furniture,East,900,
7,Priya,28,Clothing,West,400,Active

Writing dataset.csv


In [8]:
df=spark.read.csv(
    "/content/dataset.csv",
    header=True,
    inferSchema=True
)

In [9]:
df.show()
df.columns

+-------+-----+---+-----------+------+-----+--------+
|user_id| name|age|   category|region|sales|  status|
+-------+-----+---+-----------+------+-----+--------+
|      1| Amit| 25|Electronics|  West|  500|  Active|
|      2| Riya| 30|  Furniture|  East|  700|    NULL|
|      3|Rohan| 17|Electronics|  West|  300|  Active|
|      4| Neha| 22|   Clothing| North|  200|Inactive|
|      5| Amit| 25|Electronics|  West|  500|  Active|
|      6|Karan| 40|  Furniture|  East|  900|    NULL|
|      7|Priya| 28|   Clothing|  West|  400|  Active|
+-------+-----+---+-----------+------+-----+--------+



['user_id', 'name', 'age', 'category', 'region', 'sales', 'status']

In [10]:
df_clean = df.dropDuplicates()

In [11]:
df_clean.show()

+-------+-----+---+-----------+------+-----+--------+
|user_id| name|age|   category|region|sales|  status|
+-------+-----+---+-----------+------+-----+--------+
|      7|Priya| 28|   Clothing|  West|  400|  Active|
|      1| Amit| 25|Electronics|  West|  500|  Active|
|      5| Amit| 25|Electronics|  West|  500|  Active|
|      3|Rohan| 17|Electronics|  West|  300|  Active|
|      4| Neha| 22|   Clothing| North|  200|Inactive|
|      2| Riya| 30|  Furniture|  East|  700|    NULL|
|      6|Karan| 40|  Furniture|  East|  900|    NULL|
+-------+-----+---+-----------+------+-----+--------+



In [12]:
df_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).show()

+-------+----+---+--------+------+-----+------+
|user_id|name|age|category|region|sales|status|
+-------+----+---+--------+------+-----+------+
|      0|   0|  0|       0|     0|    0|     2|
+-------+----+---+--------+------+-----+------+



In [13]:
age_df=df_clean.filter(
    col("age").between(18,30)
)

age_df.show()

+-------+-----+---+-----------+------+-----+--------+
|user_id| name|age|   category|region|sales|  status|
+-------+-----+---+-----------+------+-----+--------+
|      7|Priya| 28|   Clothing|  West|  400|  Active|
|      1| Amit| 25|Electronics|  West|  500|  Active|
|      5| Amit| 25|Electronics|  West|  500|  Active|
|      4| Neha| 22|   Clothing| North|  200|Inactive|
|      2| Riya| 30|  Furniture|  East|  700|    NULL|
+-------+-----+---+-----------+------+-----+--------+



In [14]:
df_clean.filter(
    col("category")=="Electronics"
).show()

+-------+-----+---+-----------+------+-----+------+
|user_id| name|age|   category|region|sales|status|
+-------+-----+---+-----------+------+-----+------+
|      1| Amit| 25|Electronics|  West|  500|Active|
|      5| Amit| 25|Electronics|  West|  500|Active|
|      3|Rohan| 17|Electronics|  West|  300|Active|
+-------+-----+---+-----------+------+-----+------+



In [15]:
df_clean.filter(
    col("region")=="West"
).show()

+-------+-----+---+-----------+------+-----+------+
|user_id| name|age|   category|region|sales|status|
+-------+-----+---+-----------+------+-----+------+
|      7|Priya| 28|   Clothing|  West|  400|Active|
|      1| Amit| 25|Electronics|  West|  500|Active|
|      5| Amit| 25|Electronics|  West|  500|Active|
|      3|Rohan| 17|Electronics|  West|  300|Active|
+-------+-----+---+-----------+------+-----+------+



In [16]:
df_clean=df_clean.withColumnRenamed(
    "sales",
    "sale_amount"
)

In [17]:
df_clean=df_clean.withColumn(
    "age",
    col("age").cast("integer")
)

In [18]:
df_clean.count()
df_clean.select(
    avg("sale_amount")
).show()

+----------------+
|avg(sale_amount)|
+----------------+
|           500.0|
+----------------+



In [19]:
category_sales=df_clean.groupBy(
    "category"
).agg(
    sum("sale_amount").alias("total_sales"),
    avg("sale_amount").alias("avg_sales")
)

category_sales.show()

+-----------+-----------+-----------------+
|   category|total_sales|        avg_sales|
+-----------+-----------+-----------------+
|Electronics|       1300|433.3333333333333|
|   Clothing|        600|            300.0|
|  Furniture|       1600|            800.0|
+-----------+-----------+-----------------+



In [21]:
final_df=(
    df
    .dropDuplicates()
    .na.fill({"status":"Unknown"})
    .filter(col("age")>=18)
    .groupBy("category")
    .agg(
        sum("sales").alias("total_sales")
    )
)

final_df.show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|       1000|
|   Clothing|        600|
|  Furniture|       1600|
+-----------+-----------+



In [24]:
final_df.toPandas().to_csv(
    "results.csv",
    index=False
)